# Problem Statement
A small manufacturing company produces two products, A and B, every week.

Each unit of Product A requires: 2 hours of machining & 1 hour of assembly

Each unit of Product B requires: 1 hour of machining & 3 hours of assembly

The factory has:

100 machining hours available per week
90 assembly hours available per week

The profit earned is: ₹40 per unit of Product A & ₹50 per unit of Product B

The company wants to decide how many units of each product to manufacture this week to maximize total profit.

Additional conditions
Fractional production is not allowed (products must be produced in whole units).
Production quantities cannot be negative.

## Model Formulation

Sets : 
$\\ P $ - set of products
$\\ R $ - set of resources

Parameters :
$\\ a_{pr} $ - amount of resource $r$ required to produce a unit of $p$
$\\ b_r $ - total amount of resource $r$ available per week
$\\ c_p $ - profit earned per unit of $p$ sold

Variables :
$\\ x_p $ - units of product p produced

Objective :
$$ 
\max_{x_p} \sum_{p \in P} c_p x_p
$$

Constraints :
$$
\sum_{p \in P} a_{pr}x_p \leq b_r \qquad \forall r \in R
\\
x_p \in \mathbb{Z}^+
$$


In [134]:
resourceUsageData = {
    ('A','machining') : 2, 
    ('A','assembly') : 1,
    ('B','machining') : 1,
    ('B','assembly') : 3
}

resourceAvailableData = {
    'machining' : 100,
    'assembly' : 90
}

profitData = {
    'A' : 40,
    'B' : 50
}

In [135]:
from pyomo.environ import *

model = ConcreteModel()

model.products = Set(initialize = ['A','B'])
model.resources = Set(initialize = ['machining','assembly'])

model.resourceUsage = Param(model.products, model.resources, initialize=resourceUsageData)
model.resourceAvailable = Param(model.resources, initialize=resourceAvailableData)
model.profit = Param(model.products, initialize=profitData)

model.x = Var(model.products, domain=NonNegativeIntegers)

model.Obj = Objective(
    expr=sum(model.profit[p]*model.x[p] for p in model.products),
    sense=maximize
)

def model_resource_cons_rule(model,resource):
    return sum(model.resourceUsage[product,resource]*model.x[product] for product in model.products) <= model.resourceAvailable[resource]

model.resourceConstraint = Constraint(
    model.resources,
    rule = model_resource_cons_rule
)


In [136]:
import time
solver = SolverFactory('highs')

elapsed_time = time.perf_counter()
results = solver.solve(model, tee=True)
elapsed_time = time.perf_counter() - elapsed_time

##### Objective
print(f" Optimal Value of the objective is {value(model.Obj)}")

print("\n")
#### Variables
print(f" In order to obtain this we need produce : ")
for product in model.products:
    print(f"    product - {product} = {value(model.x[product])} units ")

print("\n")
#### Constraints
for resource in model.resourceConstraint:
    print(f" For Resource --> {resource} : {model.resourceConstraint[resource].expr}")
    usage = sum(model.resourceUsage[product,resource]*model.x[product] for product in model.products)
    print(f"    Usage = {value(usage)}")
    print(f"    Available = {model.resourceAvailable[resource]}")
    if model.resourceConstraint[resource].uslack() == 0:
        print("    ## This is a binding constraint ##")
    else:
        print(f"    we have a slack = {model.resourceConstraint[resource].uslack()} ")

print("\n")
### Solver
print("Solver results ::")
print(f"time taken : {elapsed_time}")
print(results)

 Optimal Value of the objective is 2480.0


 In order to obtain this we need produce : 
    product - A = 42.0 units 
    product - B = 16.0 units 


 For Resource --> machining : 2*x[A] + x[B]  <=  100
    Usage = 100.0
    Available = 100
    ## This is a binding constraint ##
 For Resource --> assembly : x[A] + 3*x[B]  <=  90
    Usage = 90.0
    Available = 90
    ## This is a binding constraint ##


Solver results ::
time taken : 0.020219656998961

Problem: 
- Lower bound: 2480.0
  Upper bound: 2480.0
  Number of objectives: 1
  Number of constraints: nan
  Number of variables: nan
  Sense: maximize
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.convergenceCriteriaSatisfied



Solving LP relaxation

In [137]:
model.x.domain = NonNegativeReals

solver = SolverFactory('highs')

elapsed_time = time.perf_counter()
results = solver.solve(model)
elapsed_time = time.perf_counter() - elapsed_time

##### Objective
print(f" Optimal Value of the objective is {value(model.Obj)}")

print("\n")
#### Variables
print(f" In order to obtain this we need produce : ")
for product in model.products:
    print(f"    product - {product} = {value(model.x[product])} units ")

print("\n")
#### Constraints
for resource in model.resourceConstraint:
    print(f" For Resource --> {resource} : {model.resourceConstraint[resource].expr}")
    usage = sum(model.resourceUsage[product,resource]*model.x[product] for product in model.products)
    print(f"    Usage = {value(usage)}")
    print(f"    Available = {model.resourceAvailable[resource]}")
    if model.resourceConstraint[resource].uslack() == 0:
        print("    ## This is a binding constraint ##")
    else:
        print(f"    we have a slack = {model.resourceConstraint[resource].uslack()} ")

print("\n")
### Solver
print("Solver results ::")
print(f"time taken : {elapsed_time}")
print(results)

 Optimal Value of the objective is 2480.0


 In order to obtain this we need produce : 
    product - A = 42.0 units 
    product - B = 16.0 units 


 For Resource --> machining : 2*x[A] + x[B]  <=  100
    Usage = 100.0
    Available = 100
    ## This is a binding constraint ##
 For Resource --> assembly : x[A] + 3*x[B]  <=  90
    Usage = 90.0
    Available = 90
    ## This is a binding constraint ##


Solver results ::
time taken : 0.00646873599907849

Problem: 
- Lower bound: 2480.0
  Upper bound: 2480.0
  Number of objectives: 1
  Number of constraints: nan
  Number of variables: nan
  Sense: maximize
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.convergenceCriteriaSatisfied



The LP relaxation has the corner points as the solution which are integer thus they are the solutions to our MILP